# Capítulo 1 — Sua primeira aplicação de IA

## Missão
Colocar um modelo para responder de verdade, medir **quanto custa** e **quanto demora**, e decidir com números se um caso de uso vale a pena. É o primeiro tijolo do projeto do curso: o **assistente de suporte da Nimbus**, uma empresa fictícia de software (os dados dela estão em `comum/dados.py`).

**No fim deste caderno você terá:** o módulo `meu_assistente/nucleo.py`, que responde uma pergunta e devolve texto, latência e tokens; e uma planilha de viabilidade de casos de uso.

**Leitura essencial (opcional, livro em inglês):**
- From Foundation Models to AI Engineering · PDF p. 36
- Planning AI Applications (Use Case Evaluation) · PDF p. 52
- The AI Engineering Stack: Three Layers · PDF p. 59

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap01_sua_primeira_aplicacao_de_ia.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [1]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

Pronto. Modelo: openai/gpt-4o-mini | modo simulado: False | seu pacote: meu_assistente


## Ideia em 30 segundos
Engenharia de IA é construir produtos **em cima** de modelos prontos (modelos de base), em vez de treinar um modelo do zero. O trabalho muda de lugar: sai a modelagem e entram avaliação, *prompts*, contexto (RAG), custo, latência e experiência do usuário. A pilha tem três camadas:
1. **aplicação** (prompt, contexto, interface, avaliação);
2. **modelo** (escolha, ajuste fino, dados);
3. **infraestrutura** (servir, escalar, monitorar).

Neste curso você trabalha quase sempre na camada de aplicação, que é onde está a maior parte das vagas.

### 1.1 **Primeira chamada.** Use `perguntar` para pedir, em 2 frases, uma explicação do que faz um engenheiro de IA. Mostre a resposta e depois `gasto()`.

In [2]:
resposta = perguntar("Em 2 frases, o que faz um engenheiro de IA?")
print(resposta)
print(gasto())

Um engenheiro de IA desenvolve algoritmos e modelos que permitem que sistemas computacionais aprendam a realizar tarefas de forma autônoma, analisando dados e identificando padrões. Além disso, ele implementa e otimiza essas soluções em aplicativos e plataformas para resolver problemas em diferentes áreas, como saúde, finanças e automação.
1 chamadas · 21 tokens de entrada · 67 de saída · US$ 0.00004


### 1.2 **Anatomia da resposta.** Com `conversar`, mande uma mensagem de sistema ("Você é o suporte da Nimbus, responda em uma frase.") e uma do usuário. Mostre: o modelo usado, o motivo de parada (`finish_reason`), os tokens de entrada e de saída (`usage`) e o texto.

In [3]:
r = conversar([
    {"role": "system", "content": "Você é o suporte da Nimbus, responda em uma frase."},
    {"role": "user", "content": "Vocês tem aplicativo para celular?"},
])
print("modelo:", r.model, "| para:", r.choices[0].finish_reason)
print("tokens:", r.usage.prompt_tokens, "entrada,", r.usage.completion_tokens, "saída")
print(r.choices[0].message.content)

modelo: openai/gpt-4o-mini | para: stop
tokens: 31 entrada, 17 saída
Sim, a Nimbus oferece um aplicativo para celular disponível em plataformas iOS e Android.


### 1.3 **Latência.** Meça com `time.perf_counter()` o tempo de 3 chamadas iguais e mostre a média e a maior. Por que as três não demoram o mesmo?

In [5]:
tempos = []
for _ in range(3):
    t0 = time.perf_counter()
    perguntar("Diga apenas: ok")
    tempos.append(time.perf_counter() - t0)
print(f"média {np.mean(tempos):.2f}s | maior {max(tempos):.2f}s")

média 0.97s | maior 1.28s


### 1.4 **Quanto custa por mês?** Escreva `custo_mensal(chamadas_dia, tokens_in, tokens_out, preco_in, preco_out)`, com os preços em US$ por **1 milhão** de tokens. Calcule para o suporte da Nimbus: 2.000 chamados por dia, ~800 tokens de entrada e ~250 de saída, comparando o `gpt-4o-mini` (0,15 / 0,60) com o `gpt-4o` (2,50 / 10,00). Os preços mudam: confira no site do OpenRouter quando for decidir de verdade.

In [6]:
def custo_mensal(chamadas_dia, tokens_in, tokens_out, preco_in, preco_out, dias=30):
    por_chamada = tokens_in / 1e6 * preco_in + tokens_out / 1e6 * preco_out
    return chamadas_dia * dias * por_chamada

for nome, pin, pout in [("gpt-4o-mini", 0.15, 0.60), ("gpt-4o", 2.50, 10.00)]:
    print(f"{nome:12} US$ {custo_mensal(2000, 800, 250, pin, pout):8.2f} por mês")

gpt-4o-mini  US$    16.20 por mês
gpt-4o       US$   270.00 por mês


### 1.5 **Vale a pena?** Monte um `DataFrame` com 4 casos de uso candidatos da Nimbus (por exemplo: responder chamados, resumir reuniões, gerar relatórios para gestores, traduzir a central de ajuda). Dê notas de 1 a 5 para: `valor` para o negócio, `risco` de erro (5 = baixo risco), `dados` disponíveis e `medir` (facilidade de medir o sucesso). Calcule uma nota ponderada (pesos à sua escolha) e ordene. Qual você faria primeiro?

In [7]:
casos = pd.DataFrame([
    {"caso": "responder chamados", "valor": 5, "risco": 3, "dados": 5, "medir": 4},
    {"caso": "resumir reuniões", "valor": 3, "risco": 4, "dados": 3, "medir": 2},
    {"caso": "relatórios para gestores", "valor": 4, "risco": 2, "dados": 4, "medir": 3},
    {"caso": "traduzir central de ajuda", "valor": 2, "risco": 5, "dados": 5, "medir": 4},
])
pesos = {"valor": 0.4, "risco": 0.2, "dados": 0.2, "medir": 0.2}
casos["nota"] = sum(casos[c] * p for c, p in pesos.items())
casos.sort_values("nota", ascending=False)

,caso,valor,risco,dados,medir,nota
0,responder chamados,5,3,5,4,4.4
3,traduzir central de ajuda,2,5,5,4,3.6
2,relatórios para gestores,4,2,4,3,3.4
1,resumir reuniões,3,4,3,2,3.0


### 1.6 **Metas antes de construir.** Defina num dicionário 3 metas mensuráveis para o assistente (por exemplo: % de respostas corretas, latência máxima, custo máximo por chamado) e mostre como uma tabela. Você vai medir essas metas nos capítulos 4 e 10.

In [8]:
metas = {
    "respostas corretas (nota do juiz >= 4)": ">= 85%",
    "latência p90": "<=4 s",
    "custo por chamado": "<= US$ 0,001",
}
pd.Series(metas, name="meta").to_frame()

,meta
respostas corretas (nota do juiz >= 4),>= 85%
latência p90,<=4 s
custo por chamado,"<= US$ 0,001"


### 1.7 **Primeiro módulo do projeto.** Escreva `nucleo.py` com:
- `SISTEMA`: o texto de sistema do assistente da Nimbus (tom educado, respostas curtas, admitir quando não sabe);
- `responder(pergunta, sistema=SISTEMA, modelo=MODELO)`: chama o modelo e devolve um dicionário com `texto`, `latencia_s`, `tokens_in`, `tokens_out` e `modelo`.

_A célula grava `meu_assistente/nucleo.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [9]:
%%writefile ../meu_assistente/nucleo.py
"""Núcleo do assistente da Nimbus: uma chamada ao modelo com medição."""
import time

from ia import MODELO, conversar

SISTEMA = ("Você é o assistente de suporte da Nimbus, um aplicativo de gestão de tarefas para equipes. "
           "Responda em português, com educação e no máximo 3 frases. "
           "Se não tiver certeza da resposta, diga que vai encaminhar para um atendente.")


def responder(pergunta: str, sistema: str = SISTEMA, modelo: str = MODELO) -> dict:
    t0 = time.perf_counter()
    r = conversar([{"role": "system", "content": sistema}, {"role": "user", "content": pergunta}], modelo=modelo)
    return {
        "texto": r.choices[0].message.content,
        "latencia_s": round(time.perf_counter() - t0, 3),
        "tokens_in": r.usage.prompt_tokens,
        "tokens_out": r.usage.completion_tokens,
        "modelo": modelo,
    }

Writing ../meu_assistente/nucleo.py


### 1.8 **Teste o núcleo.** Rode `responder` nas 3 primeiras perguntas de `PERGUNTAS` e mostre um `DataFrame` com a pergunta, o texto (cortado em 80 caracteres), a latência e os tokens. As respostas estão certas? (Por enquanto o modelo **não conhece** a Nimbus: ele vai inventar ou dizer que não sabe. O RAG do capítulo 6 resolve isso.)

In [10]:
from gabarito_assistente.nucleo import responder
linhas = []
for p in PERGUNTAS[:3]:
    r = responder(p["pergunta"])
    linhas.append({"pergunta": p["pergunta"], "resposta": r["texto"][:80], "latencia_s": r["latencia_s"],
                   "tokens": r["tokens_in"] + r["tokens_out"]})
pd.DataFrame(linhas)

,pergunta,resposta,latencia_s,tokens
0,Quanto custa o plano Equipe?,Vou encaminhar sua pergunta para um atendente ...,2.579,92
1,Tem desconto se eu pagar por ano?,"Sim, oferecemos desconto para pagamento anual....",1.881,124
2,Quantos usuários posso ter no plano grátis?,"No plano grátis da Nimbus, você pode ter até 5...",1.338,98


## Fechamento

- O que eu construí hoje: Aprendi a medir o custo de um agente, quantidade tokens usados e valores.
- O que me surpreendeu: a diferença de valores dos modelos.
- O que quero testar com o modelo de verdade (fora do modo simulado): Quero aprender a economizar na hora de desenvolver agentes.
